In [51]:
import pandas as pd

# ---- INPUT YOUR CSV FILEPATHS HERE ----
csv1 = "/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/manually parsed v2/01-PE-Test-V2.csv"
csv2 = "/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/ntbk output/forensic_features.csv"

# ---- LOAD FILES ----
df1 = pd.read_csv(csv1)
df2 = pd.read_csv(csv2)

# ---- GET COLUMN NAMES & DTYPES ----
cols1 = pd.DataFrame({
    "column_name": df1.columns,
    "Manually Parsed": df1.dtypes.astype(str).values
})

cols2 = pd.DataFrame({
    "column_name": df2.columns,
    "Notebook Output": df2.dtypes.astype(str).values
})

# ---- MERGE TO COMPARE ----
comparison = cols1.merge(cols2, on="column_name", how="outer")

# ---- FLAGS ----
comparison["name_in_both"] = comparison["Manually Parsed"].notna() & comparison["Notebook Output"].notna()
comparison["dtype_match"] = comparison["Manually Parsed"] == comparison["Notebook Output"]

# ---- SORT ----
comparison = comparison.sort_values("column_name")

# ---- EXPORT TO CSV ----
comparison.to_csv("column_dtype_comparison.csv", index=False)


# ---- HIGHLIGHTING FUNCTION ----
def highlight_diff(row):
    styles = []
    for col in row.index:
        if col == "dtype_match":
            # highlight dtype matching
            if row["dtype_match"]:
                styles.append("background-color: #d4f7d4; font-weight: bold;")  # green
            else:
                styles.append("background-color: #f7d4d4; font-weight: bold;")  # red
        elif col == "name_in_both":
            if row["name_in_both"]:
                styles.append("background-color: #d4f7d4; font-weight: bold;")
            else:
                styles.append("background-color: #f7d4d4; font-weight: bold;")
        else:
            styles.append("")  # no highlight for other cells
    return styles

# ---- SHOW WITH HIGHLIGHTING ----
comparison.style.apply(highlight_diff, axis=1)




/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_77651/3656039225.py:8: DtypeWarning: Columns (3,6,15,16,17,18,19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(csv1)
/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_77651/3656039225.py:9: DtypeWarning: Columns (3,6,15,16,17,18,19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(csv2)


,column_name,Manually Parsed,Notebook Output,name_in_both,dtype_match
0,accessed_time_changed_to_past,bool,bool,True,True
1,accessed_time_delta_days,float64,float64,True,True
2,copied_from_file,bool,bool,True,True
3,creation_time_changed_to_past,bool,bool,True,True
4,cross_artifact_validation_score,int64,int64,True,True
5,event_frequency_per_case,int64,int64,True,True
6,event_frequency_per_file,int64,int64,True,True
7,event_vs_modified_after_days,float64,float64,True,True
8,events_in_1min_window,int64,int64,True,True
9,events_in_5min_window,int64,int64,True,True


In [52]:
import pandas as pd

# ---- INPUT CSV PATHS ----
csv1 = "data/prototype_output/01-PE/manually parsed/01-PE-Test.csv"
csv2 = "/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/notebook output/forensic_features.csv"

# ---- LOAD CSVs ----
df1 = pd.read_csv(csv1, low_memory=False)
df2 = pd.read_csv(csv2, low_memory=False)

# ---- ALIGN COLUMNS BY NAME ----
common_cols = df1.columns.intersection(df2.columns)
df1_aligned = df1[common_cols].copy()
df2_aligned = df2[common_cols].copy()

# ---- RESET INDEX ----
df1_aligned.reset_index(drop=True, inplace=True)
df2_aligned.reset_index(drop=True, inplace=True)

# ---- AUTOMATIC DATETIME DETECTION AND NORMALIZATION ----
for col in common_cols:
    try:
        df1_aligned[col] = pd.to_datetime(df1_aligned[col], errors='raise')
        df2_aligned[col] = pd.to_datetime(df2_aligned[col], errors='raise')

        # Truncate seconds so 15:12 and 15:12:25 match
        df1_aligned[col] = df1_aligned[col].dt.floor("min")
        df2_aligned[col] = df2_aligned[col].dt.floor("min")
    except (ValueError, TypeError):
        # Non-datetime column → compare as strings
        df1_aligned[col] = df1_aligned[col].astype(str)
        df2_aligned[col] = df2_aligned[col].astype(str)

print(f"CSV1 shape: {df1_aligned.shape}, CSV2 shape: {df2_aligned.shape}")

# ---- FIND MISMATCHES ----
mismatches = df1_aligned != df2_aligned

# ---- COUNT MISMATCHES PER COLUMN ----
mismatch_counts = mismatches.sum().reset_index()
mismatch_counts.columns = ["column_name", "num_mismatches"]

print("\n==== MISMATCH COUNT PER COLUMN ====")
print(mismatch_counts[mismatch_counts["num_mismatches"] > 0])

# ---- EXPORT MISMATCH COUNTS ----
mismatch_counts.to_csv("column_mismatch_counts.csv", index=False)
print("\nMismatch counts exported → column_mismatch_counts.csv")

# ---- EXPORT FULL MISMATCH DETAILS ----
if mismatches.any().any():
    print("\nData mismatch found! Creating detailed diff CSV...")

    df_diff = df1_aligned.copy()

    for col in df_diff.columns:
        df_diff[col] = df_diff[col].where(
            ~mismatches[col],
            df1_aligned[col].astype(str) + " != " + df2_aligned[col].astype(str)
        )
    
    # Export mismatch details
    df_diff.to_csv("csv_value_mismatches.csv", index=False)
    print("Full mismatch details exported → csv_value_mismatches.csv")

    # Show mismatches in notebook
    display(df_diff[mismatches])
else:
    print("All rows and columns match exactly!")


/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_77651/1237335609.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1_aligned[col] = pd.to_datetime(df1_aligned[col], errors='raise')
/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_77651/1237335609.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1_aligned[col] = pd.to_datetime(df1_aligned[col], errors='raise')
/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_77651/1237335609.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1_aligned[col] = pd.to_datetime(df1_aligned[col], error

CSV1 shape: (24204, 54), CSV2 shape: (24204, 54)

==== MISMATCH COUNT PER COLUMN ====
                          column_name  num_mismatches
0                           eventtime             175
1                        eventtime_dt             175
2                              lf_lsn           23776
3                            lf_event            1614
4                            filename           19142
5                            filepath           12748
6                       lf_target_vcn            1849
7                    lf_cluster_index           23776
8                           merge_key           19718
9                             usn_usn             189
10                     usn_event_info           13529
11          usn_file_reference_number           19093
12   usn_parent_file_reference_number           11881
13                             source            1616
14                       is_tunneling             472
15            lf_creation_time_before           24

,eventtime,eventtime_dt,lf_lsn,lf_event,filename,filepath,lf_target_vcn,lf_cluster_index,merge_key,usn_usn,...,usn_file_closed,usn_complete_manipulation_pattern,path_depth,event_vs_modified_after_days,cross_artifact_validation_score,timestamp_manipulation_pattern_score,file_system_tunneling_confidence,has_attribute_change,has_timestamp_copied_from_file,zero_nanoseconds_logfile
0,NaN,NaN,NaT != NaT,NaN,dsregcmd.exe.mui != dsuiwiz.dll.mui,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,NaT != NaT,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
1,NaN,NaN,NaT != NaT,NaN,dsregcmd.exe.mui != dwmredir.dll.mui,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,NaT != NaT,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
2,NaN,NaN,NaT != NaT,NaN,dsregtask.dll.mui != DWrite.dll.mui,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,NaT != NaT,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,NaN,NaN,NaT != NaT,NaN,dsregtask.dll.mui != DWrite.dll.mui,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,NaT != NaT,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
4,NaN,NaN,NaT != NaT,NaN,dsrolesrv.dll.mui != DWWIN.exe.mui,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,NaT != NaT,\Program Files\WindowsApps\Microsoft.LanguageE...,NaN,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24199,NaT != NaT,NaT != NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT != NaT,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
24200,NaT != NaT,NaT != NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT != NaT,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
24201,NaT != NaT,NaT != NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT != NaT,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
24202,NaT != NaT,NaT != NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT != NaT,...,NaN,NaN,NaT,NaT != NaT,NaT,NaT,NaT,NaT,NaT,NaT
